# Sleep English · Colab 一键运行

从 GitHub 拉**最新**代码 → 装依赖 → 起 Web 控制台（公网链接）→ 出片 → 成品归档到 Drive。

本 notebook 只是薄引导层：安装/配置/启动逻辑都在仓库 `colab/` 下的脚本里，
所以仓库一更新，重跑「第 1 格」就自动用上新版本。

**首次使用**
1. 菜单 `Runtime → Change runtime type` 选 **T4 GPU**（Kokoro TTS 快很多；CPU 也能跑，只是慢）
2. 左侧 🔑 **Secrets** 建两个键：
   - `GEMINI_API_KEY`（或 `OPENAI_API_KEY`）→ LLM 密钥，生成脚本用
   - `GITHUB_TOKEN` → **仓库是私有的才需要**。GitHub → Settings → Developer settings → Personal access tokens，
     给 repo 读权限。缺了它，第 1 格的 `git clone` 会以 404/401 失败
3. 从上往下依次运行；「启动 Web 控制台」那格会打印一条带令牌的公网链接，点开就是完整控制台

**已知落差**（详见 `colab/README.md`）
- 预生成脚本成品已入库（`ai_scripts/` 100 个 + `ai_scripts_hot/` 117 个），克隆完就能在控制台里选；
  生成过程的中间件（`.llm_cache`、`_parts`、`_review`…）被 `.gitignore` 排除，Colab 上也用不着
- `configs/` 下带密钥的配置被 `.gitignore` 排除 → 每次开机由「写入配置」那格从 Secrets 重新生成
- Qwen / MOSS 音色引擎、BGM 混音、片头片尾素材库都依赖本机大文件 → Colab 上一律 Kokoro + Pillow 回落

In [ ]:
# ===================== 参数区（一般只改这里）=====================
REPO_URL = "https://github.com/collinsgraciano/sleep_english.git"
BRANCH   = "master"
REPO_DIR = "/content/sleep_english"
# 私有仓库必填：Secrets 里存 GitHub PAT 的**键名**（公开仓库留空 ""）
SECRET_GITHUB_TOKEN = "GITHUB_TOKEN"

# 渲染目录固定在容器本地（上千个卡片/音频小文件，写 Drive 会慢到不可用）；
# Drive 只用来归档成品（mp4 / 缩略图 / 字幕 / 元数据），重启不丢
LOCAL_OUTPUT_DIR = "/content/sleep_english_output"
ARCHIVE_TO_DRIVE = True
DRIVE_DIR        = "sleep_english_colab"   # MyDrive 下的子目录名

# LLM 通道（生成脚本用）：填 Secrets 里的**键名**，不是密钥本身
#   用 Gemini        → SECRET_GEMINI_API_KEY = "GEMINI_API_KEY"，其余留空
#   用中转/自建端点   → SECRET_OPENAI_API_KEY = "OPENAI_API_KEY"，并填 OPENAI_BASE_URL / OPENAI_MODEL
SECRET_GEMINI_API_KEY = "GEMINI_API_KEY"
SECRET_OPENAI_API_KEY = ""
OPENAI_BASE_URL = ""                     # 例：https://your-endpoint/v1
OPENAI_MODEL    = ""                     # 例：gemini-3.6-flash
GEMINI_MODEL    = ""                     # 留空 = 用仓库配置里的默认模型

# 本期视频参数（Web 控制台里也能改，这里只决定首次落盘的配置）
TOPIC        = ""                        # 留空 = 从主题库随机挑
SLEEP_PAIRS  = 200                       # A/B 组数：200 组 = 400 句 ≈ 100 分钟；冒烟用 10
NO_4K        = True                      # Colab 上 4K 放大很慢，默认关
CHANNEL_NAME = ""                        # 留空 = 用仓库配置里的频道名
# ===============================================================
print("参数区已加载")

## 1. 拉最新代码 + 挂载 Drive + 写 env 文件

In [ ]:
import os, secrets, shlex, subprocess
from urllib.parse import quote, urlparse, urlunparse

def _secret(key_name):
    """从 Colab Secrets 取密钥；非 Colab 环境回落同名环境变量。"""
    if not key_name:
        return ""
    try:
        from google.colab import userdata
        return (userdata.get(key_name) or "").strip()
    except Exception:
        return os.environ.get(key_name, "").strip()

_GH_TOKEN = _secret(SECRET_GITHUB_TOKEN)
_MASKS = [m for m in (_GH_TOKEN, quote(_GH_TOKEN, safe="")) if m]

def _mask(text):
    # git 失败时会把整条 URL（含令牌）回显出来，打印前统一打码
    for m in _MASKS:
        text = text.replace(m, "****")
    return text

def run(cmd, check=False):
    print("$", _mask(" ".join(cmd)))
    r = subprocess.run(cmd, text=True, capture_output=True)
    for out in (r.stdout, r.stderr):
        if out.strip():
            print(_mask(out.strip()))
    if check and r.returncode != 0:
        raise SystemExit(f"命令失败（{r.returncode}）")
    return r

def _auth_url(url):
    """私有仓库把 PAT 放进 URL 的 userinfo 段；只活在内存里，不写进 .git/config。"""
    if not _GH_TOKEN:
        return url
    p = urlparse(url)
    return urlunparse(p._replace(netloc=f"{quote(_GH_TOKEN, safe='')}@{p.netloc}"))

# 已克隆过就快进更新，否则浅克隆
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    r = run(["git", "-C", REPO_DIR, "pull", "--ff-only", _auth_url(REPO_URL), BRANCH])
    if r.returncode != 0:
        raise SystemExit("git pull 失败（Colab 里的代码被改过？）。菜单 Runtime → Restart runtime 后重跑本格，即可拿到干净的最新代码。")
else:
    r = run(["git", "clone", "--depth", "1", "-b", BRANCH, _auth_url(REPO_URL), REPO_DIR])
    if r.returncode != 0:
        raise SystemExit("克隆失败。仓库是私有的话，去左侧 🔑 Secrets 建 GITHUB_TOKEN（GitHub PAT，给 repo 读权限）再重跑本格；"
                         "公开仓库则检查参数区的 REPO_URL / BRANCH。")
    if _GH_TOKEN:
        run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
print("拉取方式：", "带令牌（私有仓库）" if _GH_TOKEN else "匿名（公开仓库）")
print("当前代码：", run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).stdout.strip())

# 输出：渲染在本地，成品归档到 Drive
OUTPUT_DIR = LOCAL_OUTPUT_DIR
ARCHIVE_DIR = ""
os.makedirs(OUTPUT_DIR, exist_ok=True)
if ARCHIVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARCHIVE_DIR = os.path.join("/content/drive/MyDrive", DRIVE_DIR, "output")
    os.makedirs(ARCHIVE_DIR, exist_ok=True)

# 变量写成 shell env 文件，后面的 ! 单元格 source 它
ACCESS_TOKEN = os.environ.get("COLAB_ACCESS_TOKEN") or secrets.token_urlsafe(12)
ENV_FILE = "/content/colab_env.sh"
env_vars = {
    "REPO_DIR": REPO_DIR, "PORT": "8766", "ENV_FILE": ENV_FILE,
    "COLAB_OUTPUT_DIR": OUTPUT_DIR, "COLAB_ARCHIVE_DIR": ARCHIVE_DIR,
    "COLAB_ACCESS_TOKEN": ACCESS_TOKEN,
    "COLAB_TTS_ENGINE": "kokoro", "COLAB_NO_4K": "1" if NO_4K else "0",
    "COLAB_SLEEP_PAIRS": str(SLEEP_PAIRS), "COLAB_TOPIC": TOPIC,
    "COLAB_CHANNEL_NAME": CHANNEL_NAME,
}
with open(ENV_FILE, "w", encoding="utf-8") as fh:
    for k, v in env_vars.items():
        fh.write(f"export {k}={shlex.quote(str(v))}\n")
os.environ.update(env_vars)
print("env 文件   →", ENV_FILE)
print("渲染目录   →", OUTPUT_DIR)
print("归档目录   →", ARCHIVE_DIR or "(未启用 Drive 归档)")

## 2. 装依赖（ffmpeg / espeak-ng / 字体 / Kokoro TTS / cloudflared）

首次约 3-6 分钟，结束会打印自检结果（torch 是否吃到 GPU、字体是否就位、预生成脚本有几个）。

In [ ]:
!source /content/colab_env.sh && bash "$REPO_DIR/colab/setup.sh"

## 3. 写入配置（密钥从 Secrets 读，不落仓库、不打印明文）

把 `configs/mode_sleep.json` 改成 Colab 可用版本：渲染目录、主题库路径、TTS 引擎、4K 开关、LLM 通道，
并清掉本机 `H:\\` 模型路径、关掉依赖本机素材的 BGM 混音与本地代理。
同时生成 `/content/colab_env_llm.sh`，命令行直接跑 pipeline.py 时 source 它。

In [ ]:
import os, subprocess

# _secret() 在「1. 拉最新代码」那格定义（本格必须在它之后跑）
gemini_key = _secret(SECRET_GEMINI_API_KEY)
openai_key = _secret(SECRET_OPENAI_API_KEY)

# LLM_ENV 会被后面的命令行出片单元格复用（subprocess env）
LLM_ENV = {}
if openai_key:
    if not (OPENAI_BASE_URL and OPENAI_MODEL):
        raise SystemExit("用自定义端点就要在参数区填 OPENAI_BASE_URL 和 OPENAI_MODEL")
    LLM_ENV = {"LLM_PROVIDER_TYPE": "openai", "OPENAI_API_KEY": openai_key,
               "OPENAI_BASE_URL": OPENAI_BASE_URL, "OPENAI_MODEL": OPENAI_MODEL}
elif gemini_key:
    LLM_ENV = {"LLM_PROVIDER_TYPE": "gemini", "GEMINI_API_KEY": gemini_key,
               "GEMINI_MODEL": GEMINI_MODEL}
else:
    print("⚠️ 没读到任何 LLM 密钥：Web UI 照样能开，但生成脚本会失败。")
    print("   去左侧 🔑 Secrets 建好键，再重跑本格。")

r = subprocess.run(["python3", os.path.join(REPO_DIR, "colab", "colab_config.py")],
                   env={**os.environ, **LLM_ENV}, text=True, capture_output=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(r.stderr.strip())
    raise SystemExit("配置写入失败")

## 4. 预热 Kokoro（首次下载模型 ≈300MB，顺便试听一句）

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.join(REPO_DIR, "pipeline"))

t0 = time.time()
from tts_engine import TTSEngine
warm_path = "/tmp/kokoro_warm.mp3"
dur = TTSEngine().synth_english("Good night. Sweet dreams. See you tomorrow.",
                                "af_sarah", warm_path, "+0%")
print(f"Kokoro 就绪：{dur:.2f}s 音频，用时 {time.time() - t0:.1f}s")

from IPython.display import Audio
Audio(warm_path)

## 5. 启动 Web 控制台（公网链接）

跑完打印一条 `https://xxxx.trycloudflare.com/?ct=令牌` —— 点开就是和本机 `localhost:8766` 一样的完整控制台
（控制台/参数配置/脚本库/运行历史/片头库/三音色页全都在）。

> **令牌闸门是必需的**：`GET /api/config` 会返回含密钥的整份配置，公开隧道不加锁等于把密钥挂到公网。
> 所以这条链接不要转发；万一泄漏，改参数区下面的 `COLAB_ACCESS_TOKEN`（或重跑第 1 格）再重启本格。

In [ ]:
!source /content/colab_env.sh && bash "$REPO_DIR/colab/serve.sh"

## 6. 不开 Web UI，直接命令行出片

适合挂着一口气跑完。中断后重跑同一格会走文件级续传（音频/卡片按文件缓存），也可显式加 `--resume`。
输出实时打印在本格里；跑完记得执行「7. 归档成品到 Drive」。

In [ ]:
# 冒烟：10 组 ≈ 几分钟，验证「LLM 脚本 → Kokoro TTS → Pillow 卡片 → ffmpeg 合成」全链路
import os, subprocess, sys

cmd = [sys.executable, "pipeline.py", "--topic", "Ordering Coffee", "--cefr", "A2",
       "--sleep-pairs", "10", "--no-4k", "--output", os.environ["COLAB_OUTPUT_DIR"]]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "pipeline"), env={**os.environ, **LLM_ENV})

In [ ]:
# 正式一期：用参数区的 TOPIC / SLEEP_PAIRS / NO_4K / CHANNEL_NAME（200 组在 T4 上约 40-90 分钟）
import os, subprocess, sys

cmd = [sys.executable, "pipeline.py", "--cefr", "A2",
       "--sleep-pairs", str(SLEEP_PAIRS), "--output", os.environ["COLAB_OUTPUT_DIR"]]
if TOPIC:
    cmd += ["--topic", TOPIC]
if NO_4K:
    cmd += ["--no-4k"]
if CHANNEL_NAME:
    cmd += ["--sleep-channel-name", CHANNEL_NAME]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "pipeline"), env={**os.environ, **LLM_ENV})

## 7. 归档成品到 Drive

只拷成品（`videos/*.mp4`、`thumbnail.jpg`、`subtitles/`、`script.json`、`youtube_metadata.json`），
跳过 `audio/ cards/ clips/ images/ tmp_sleep_blocks/` 这些体积大又只在续传时有用的中间素材。
Web 控制台跑完的期次也一样在这里归档。

In [ ]:
import os, shutil, sys, time
sys.path.insert(0, REPO_DIR)
from app.config_manager import iter_run_dirs   # 复用项目自己的 run 发现逻辑（新旧布局都认）

ARCHIVE_DIR = os.environ.get("COLAB_ARCHIVE_DIR", "")
OUTPUT_DIR = os.environ["COLAB_OUTPUT_DIR"]
SKIP_DIRS = {"audio", "cards", "clips", "images", "tmp_sleep_blocks"}

if not ARCHIVE_DIR:
    print("未启用 Drive 归档（参数区 ARCHIVE_TO_DRIVE = False）")
else:
    def _ignore(_dir, names):
        return [n for n in names if n in SKIP_DIRS or n.startswith(".")]

    runs = iter_run_dirs(OUTPUT_DIR)
    if not runs:
        print(f"{OUTPUT_DIR} 下还没有 run")
    for src in runs:
        src = str(src)
        videos = os.path.join(src, "videos")
        mp4s = [f for f in os.listdir(videos) if f.endswith(".mp4")] if os.path.isdir(videos) else []
        dst = os.path.join(ARCHIVE_DIR, os.path.basename(src))
        t0 = time.time()
        shutil.copytree(src, dst, dirs_exist_ok=True, ignore=_ignore)
        flag = "OK " if mp4s else "!! 无成品 mp4（可能没跑完）"
        print(f"{flag} {os.path.basename(src)[:58]} | mp4 {len(mp4s)} | {time.time() - t0:.0f}s")
    print(f"\n归档目录：{ARCHIVE_DIR}（{len(runs)} 个 run）")

## 8. 可选：从 Drive 补同步还没入库的预生成脚本

成品脚本（`manifest.json` + `NNN_主题/script.json`）**已经在 Git 里**（`ai_scripts/` 100 个 + `ai_scripts_hot/` 117 个），
克隆完控制台「预生成脚本」下拉直接能选 —— 正常情况下**跳过本格**。

只有一种情况需要它：你本机刚写完新批次、还没 push。把这两个目录上传到
`MyDrive/sleep_english_colab/` 下再跑本格，会覆盖进克隆目录（Drive 小文件拷贝 1-3 分钟）。
生成过程的中间件（`.llm_cache`、`_parts`、`_review`、`_audit`…）传上去也没用，管理台不读它们。

In [ ]:
import os, shutil, time

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

src_root = os.path.join("/content/drive/MyDrive", DRIVE_DIR)
for name in ("ai_scripts", "ai_scripts_hot"):
    src = os.path.join(src_root, name)
    if not os.path.isdir(src):
        print(f"跳过 {name}：Drive 里没有 {src}")
        continue
    dst = os.path.join(REPO_DIR, name)
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    n = len([d for d in os.listdir(dst) if not d.startswith("_")])
    print(f"{name} → {dst}（{n} 个主题，用时 {time.time() - t0:.0f}s）")
print("同步完刷新 Web 控制台即可在「预生成脚本」下拉里看到。")

## 9. 运维：日志 / 成品清单 / 停止

In [ ]:
!tail -n 80 /content/colab_logs/uvicorn.log

In [ ]:
# 成品清单：本地渲染目录 + Drive 归档目录里的 mp4
import os

for label, root in (("本地", os.environ["COLAB_OUTPUT_DIR"]),
                    ("Drive", os.environ.get("COLAB_ARCHIVE_DIR", ""))):
    if not root or not os.path.isdir(root):
        continue
    hits = []
    for dirpath, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ("audio", "cards", "clips", "images", "tmp_sleep_blocks")]
        for f in files:
            if f.endswith(".mp4"):
                p = os.path.join(dirpath, f)
                hits.append((os.path.getsize(p), p))
    print(f"--- {label}：{root}（{len(hits)} 个 mp4）")
    for size, p in sorted(hits, reverse=True)[:20]:
        print(f"  {size / 1024 / 1024:9.1f} MB  {os.path.relpath(p, root)}")

In [ ]:
!pkill -f "uvicorn secure_gate:app"; pkill -f "cloudflared tunnel"; echo "已停止 Web 服务与隧道"